# MBL Dwn0-Cmp1: joint cycle waveform autoencoder

One example is **all available PV waveforms from the same cycle**. The input is
`(batch, 36 PV channels, native samples)`, with a fixed alphabetical PV order.
The 5,032 / 35,221 recorded samples are retained. There is no conversion to
256 points, decimation, block averaging, or PCA before the model.

The convolutional encoder mixes PV channels to learn their joint behavior.
Missing PVs and any unobserved trailing samples have a validity mask: they are
excluded from normalization, loss, and anomaly scores. A reconstruction error
is calculated for each observed PV, and their mean is the cycle anomaly score.

Set `RUN_TRAINING = True` to train this new model, then run all cells.
Training also calibrates the threshold and evaluates the held-out custom test
cycles. To reevaluate saved weights, set only `RUN_EVALUATION = True`.
Both switches default to `False`; opening/running the notebook then displays
saved cycle-model results if available. Results are saved separately in
`mbl_cycle_autoencoder`; previous per-waveform checkpoints are not reused.

Terminal training without Jupyter: `python -u train_mbl_cycle_autoencoder.py --device cuda`.
Use `--epochs`, `--batch-size` (whole cycles), and `--output` to configure a run.

The input remains `mbl_dwn0_cmp1_normal_without_official_holdout.pkl`, containing
the 36 selected MBL Dwn0-Cmp1 PVs. All 42 official holdout cycle IDs are excluded.
Upstream normal candidates were selected by full-data PCA+kNN, so this is not
an independent labelled anomaly benchmark. The official 256-point score does
not evaluate this joint native-waveform model.


In [1]:
from pathlib import Path
from types import SimpleNamespace
import copy
import hashlib
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import Image, display

ROOT = Path.cwd()  # Directory containing this notebook and the dataset
OUTPUT = ROOT / "mbl_cycle_autoencoder"
RUN_TRAINING = False
RUN_EVALUATION = False
args = SimpleNamespace(
    input=str(ROOT / "mbl_dwn0_cmp1_normal_without_official_holdout.pkl"),
    output=str(OUTPUT), epochs=160, patience=10, batch_size=4,
    lr=1e-3, seed=42, latent=64,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("Training:", RUN_TRAINING, "Recompute evaluation:", RUN_EVALUATION)
print("Results directory:", OUTPUT)


Training: False Recompute evaluation: False
Results directory: /lustre/work/pdl18409/shared/Anomaly_Hunter/mbl_cycle_autoencoder


## 1. Group native waveforms by cycle

PV names form fixed channels; their order never depends on row order. All
observed PVs from a cycle enter the model together. Empty/nonfinite waveforms,
unknown PVs, and duplicate `(cycle_id, PV)` captures raise an error instead of
silently losing data. Missing PVs are zero-filled after normalization and
accompanied by a mask. Different waveform lengths are right-padded only;
the original sample indices and every recorded value are preserved.


In [2]:
def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(path)


def prepare_cycles(frame, pvs=None):
    required = {"cycle_id", "pv", "data_array"}
    if not required.issubset(frame):
        raise ValueError(f"Missing columns: {sorted(required - set(frame.columns))}")
    if frame.empty or frame.cycle_id.isna().any() or frame.pv.isna().any():
        raise ValueError("Nonempty input with cycle_id and pv is required")
    frame = frame.copy()
    frame["pv"] = frame.pv.str.replace(r"-(?:PMData|DODData)$", "", case=False, regex=True)
    pvs = sorted(frame.pv.unique()) if pvs is None else list(pvs)
    unknown = set(frame.pv) - set(pvs)
    if unknown:
        raise ValueError(f"PVs absent from training: {sorted(unknown)}")
    if frame.duplicated(["cycle_id", "pv"]).any():
        raise ValueError("Duplicate (cycle_id, PV) captures; choose a capture explicitly")
    cycles = []
    for cycle_id, group in frame.groupby("cycle_id", sort=True):
        waveforms = {}
        for pv, values in zip(group.pv, group.data_array):
            values = np.asarray(values, dtype=np.float32)
            if values.ndim != 1 or len(values) < 32 or not np.isfinite(values).all():
                raise ValueError(f"cycle={cycle_id}, pv={pv}: expected a finite 1-D waveform of at least 32 samples")
            waveforms[pv] = values
        record = dict(cycle_id=int(cycle_id), waveforms=waveforms,
                      length=max(map(len, waveforms.values())))
        for column in ["cycle_ts", "beam_mode"]:
            if column in group:
                values = group[column].dropna().unique()
                if len(values) != 1:
                    raise ValueError(f"cycle={cycle_id}: inconsistent {column}")
                record[column] = str(values[0])
        cycles.append(record)
    return cycles, pvs


def pack_cycles(cycles, pvs, means, stds, device="cpu"):
    if not cycles:
        raise ValueError("An empty cycle batch cannot be packed")
    width = max(c["length"] for c in cycles)
    values = np.zeros((len(cycles), len(pvs), width), dtype=np.float32)
    mask = np.zeros(values.shape, dtype=bool)
    lengths = np.zeros((len(cycles), len(pvs)), dtype=np.int64)
    channels = {pv: j for j, pv in enumerate(pvs)}
    for i, record in enumerate(cycles):
        for pv, waveform in record["waveforms"].items():
            j = channels[pv]
            length = len(waveform)
            values[i, j, :length] = (waveform - means[j]) / stds[j]
            mask[i, j, :length] = True
            lengths[i, j] = length
    return (torch.from_numpy(values).to(device),
            torch.from_numpy(mask).to(device), lengths)


## 2. Independent partitions and training-only normalization

Archives connected by shared cycle IDs stay in the same partition. No cycle
contributes some PVs to training and other PVs to validation or test.
Training-only mean and standard deviation are fitted per PV on its observed
native samples. The prepared dataset has incomplete cycles as well as cycles
containing all 36 PVs; the coverage is included in saved predictions.


In [3]:
def grouped_splits(meta, seed):
    """Keep archives connected by shared cycle IDs in the same partition."""
    parents = {str(a): str(a) for a in meta.source_archive.unique()}
    def root(a):
        while parents[a] != a:
            parents[a] = parents[parents[a]]
            a = parents[a]
        return a
    for _, rows in meta.groupby("cycle_id", sort=False):
        archives = rows.source_archive.unique()
        for archive in archives[1:]:
            parents[root(archive)] = root(archives[0])
    groups = meta.source_archive.map(root)
    group_modes = meta.assign(group=groups).groupby("group").beam_mode.agg(lambda s: "|".join(sorted(s.unique())))
    rng = np.random.default_rng(seed)
    assignment = {}
    for _, categories in group_modes.groupby(group_modes):
        keys = np.array(sorted(categories.index), dtype=object)
        rng.shuffle(keys)
        if len(keys) < 4:
            raise ValueError("At least four independent capture groups per mode are required")
        size = max(1, round(len(keys)*.1))
        for split, members in [("test", keys[:size]), ("calibration", keys[size:2*size]),
                               ("validation", keys[2*size:3*size]), ("train", keys[3*size:])]:
            assignment.update({key: split for key in members})
    return groups.map(assignment)

class CycleData:
    def __init__(self, args):
        self.args = args
        self.output = Path(args.output)
        self.device = torch.device(args.device)
        self.rng = np.random.default_rng(args.seed)
        frame = pd.read_pickle(args.input)
        required = {"cycle_id", "cycle_ts", "pv", "data_array", "beam_mode", "source_archive"}
        if not required.issubset(frame):
            raise ValueError(f"Missing columns: {sorted(required - set(frame.columns))}")
        sample_root = ROOT / "official_anomaly_reference/data/sample"
        sample = pd.read_parquet(sample_root / "ess_llrf_sample.parquet", columns=["cycle_id"])
        holdout = json.loads((sample_root / "holdout_ids.json").read_text())["holdout_idx"]
        official_cycles = sorted(set(map(int, sample.iloc[holdout].cycle_id)))
        if frame.cycle_id.isin(official_cycles).any():
            raise ValueError("Official evaluation cycles are present in the input")
        frame = frame.copy()
        frame["source_pv"] = frame.pv
        frame["pv"] = frame.pv.str.replace(r"-(?:PMData|DODData)$", "", case=False, regex=True)
        if not frame.pv.str.fullmatch(r"MBL.*:Dwn0-Cmp1SDS", na=False).all():
            raise ValueError("Input includes PVs outside MBL Dwn0-Cmp1")
        self.records, self.pvs = prepare_cycles(frame)
        meta = frame[["cycle_id", "cycle_ts", "pv", "source_pv", "beam_mode", "source_archive"]].copy()
        meta.insert(0, "original_row", frame.index.to_numpy())
        meta["length"] = frame.data_array.map(len)
        meta["split"] = grouped_splits(meta, args.seed)
        assert meta.groupby("cycle_id").split.nunique().max() == 1
        assert meta.groupby("source_archive").split.nunique().max() == 1
        split_by_cycle = meta.groupby("cycle_id").split.first()
        self.meta = pd.DataFrame([
            dict(cycle_id=c["cycle_id"], cycle_ts=c["cycle_ts"], beam_mode=c["beam_mode"],
                 length=c["length"], observed_pvs=len(c["waveforms"]),
                 missing_pvs=len(self.pvs) - len(c["waveforms"]),
                 split=split_by_cycle.loc[c["cycle_id"]]) for c in self.records
        ])
        self.waveform_meta = meta.reset_index(drop=True)
        self.lengths = self.meta.length.to_numpy()
        self.splits = {s: np.flatnonzero(self.meta.split.eq(s))
                       for s in ["train", "validation", "calibration", "test"]}
        self.means, self.stds = self.fit_normalization()
        identity_columns = ["original_row", "cycle_id", "pv", "length", "split", "source_archive"]
        identity = hashlib.sha256(meta[identity_columns].to_csv(index=False).encode()).hexdigest()
        self.config = dict(format="joint_native_cycle_v1", pvs=self.pvs, latent=args.latent,
                           mean=self.means.tolist(), std=self.stds.tolist(), seed=args.seed,
                           split_sha256=identity, official_holdout_cycle_ids=official_cycles,
                           official_holdout_cycle_overlap=0,
                           input_shape="batch, PV channels, native samples",
                           input_resampling=False, missing_values="mask; excluded from loss and scores",
                           cycle_score="Mean of per-PV normalized MSE over observed PVs",
                           threshold_rule="Calibration cycle score median + 5 * population std (ddof=0)",
                           cycle_counts={s: len(v) for s, v in self.splits.items()},
                           input=str(Path(args.input).resolve()))
        payload = json.dumps(self.config, sort_keys=True).encode()
        self.config["signature"] = hashlib.sha256(payload).hexdigest()
        previous = self.output / "configuration.json"
        if previous.exists() and json.loads(previous.read_text())["signature"] != self.config["signature"]:
            raise ValueError("Data/preprocessing/model changed; choose a new output directory")
        print(self.meta.groupby("split").agg(cycles=("cycle_id", "size"),
                                            min_pvs=("observed_pvs", "min"), max_pvs=("observed_pvs", "max")))
        print(f"Input: {len(self.pvs)} fixed PV channels; native lengths {sorted(set(meta.length))}")

    def fit_normalization(self):
        channels = {pv: j for j, pv in enumerate(self.pvs)}
        count = np.zeros(len(self.pvs), dtype=np.int64)
        mean = np.zeros(len(self.pvs), dtype=np.float64)
        m2 = np.zeros(len(self.pvs), dtype=np.float64)
        for index in self.splits["train"]:
            for pv, waveform in self.records[index]["waveforms"].items():
                j = channels[pv]
                values = waveform.astype(np.float64)
                n, batch_mean = len(values), values.mean()
                delta = batch_mean - mean[j]
                new_count = count[j] + n
                m2[j] += ((values - batch_mean)**2).sum() + delta**2 * count[j] * n / new_count
                mean[j] += delta * n / new_count
                count[j] = new_count
        if (count == 0).any():
            raise ValueError("Every PV must have observed training samples")
        std = np.sqrt(m2 / count)
        return mean.astype(np.float32), np.where(std >= 1e-6, std, 1.).astype(np.float32)

    def save_preprocessing(self):
        self.output.mkdir(parents=True, exist_ok=True)
        write_json(self.output / "configuration.json", self.config)
        self.meta.to_csv(self.output / "cycle_splits.csv", index=False)
        self.waveform_meta.to_csv(self.output / "sample_splits.csv", index=False)
        pd.DataFrame(dict(pv=self.pvs, mean=self.means, std=self.stds)).to_csv(
            self.output / "normalization_native.csv", index=False)

    def batches(self, indices, shuffle=False):
        batches = []
        for length in sorted(set(self.lengths[indices])):
            group = np.asarray(indices)[self.lengths[indices] == length].copy()
            if shuffle:
                self.rng.shuffle(group)
            batches.extend(group[start:start + self.args.batch_size]
                           for start in range(0, len(group), self.args.batch_size))
        if shuffle:
            self.rng.shuffle(batches)
        return batches

    def pack(self, indices):
        return pack_cycles([self.records[i] for i in indices], self.pvs,
                           self.means, self.stds, self.device)


## 3. Joint multichannel autoencoder and masked reconstruction error

The first convolution receives all PV channels and their validity masks at
once. A 64-dimensional latent vector represents the entire cycle. The decoder
reconstructs all PV channels at the input sample count. Pooling/interpolation
operates only on internal features; input waveforms are never resampled.

For each PV, MSE is averaged over its observed samples. Loss is the mean of
these PV errors per cycle, then the mean over cycles in the batch. Missing
channels and padded positions do not contribute. The model uses the waveforms
and masks; beam mode is retained as evaluation metadata.


In [4]:
class CycleWaveformAE(nn.Module):
    def __init__(self, pvs, latent=64):
        super().__init__()
        self.pvs = pvs
        self.encoder = nn.ModuleList([
            nn.Conv1d(2 * pvs, 32, 8, 4, 2),
            nn.Conv1d(32, 64, 4, 2, 1),
            nn.Conv1d(64, 64, 4, 2, 1),
        ])
        self.pool = nn.AdaptiveAvgPool1d(32)
        self.to_latent = nn.Linear(64 * 32, latent)
        self.from_latent = nn.Linear(latent, 64 * 32)
        self.decoder = nn.ModuleList([
            nn.ConvTranspose1d(64, 64, 4, 2, 1),
            nn.ConvTranspose1d(64, 32, 4, 2, 1),
            nn.ConvTranspose1d(32, pvs, 8, 4, 2),
        ])

    def forward(self, values, mask):
        if values.ndim != 3 or values.shape[1] != self.pvs or mask.shape != values.shape:
            raise ValueError("Expected matching (batch, trained PV channels, samples) tensors")
        x = torch.cat([values.masked_fill(~mask, 0), mask.to(values.dtype)], dim=1)
        sizes = []
        for layer in self.encoder:
            sizes.append(x.shape[-1])
            x = F.gelu(layer(x))
        encoded_length = x.shape[-1]
        latent = self.to_latent(self.pool(x).flatten(1))
        x = self.from_latent(latent).reshape(-1, 64, 32)
        x = F.interpolate(x, size=encoded_length, mode="linear", align_corners=False)
        for j, (layer, length) in enumerate(zip(self.decoder, reversed(sizes))):
            x = layer(x, output_size=(values.shape[0], layer.out_channels, length))
            if j < len(self.decoder) - 1:
                x = F.gelu(x)
        return x


def reconstruction_errors(prediction, target, mask):
    counts = mask.sum(dim=-1)
    observed = counts > 0
    squared_error = (prediction.float() - target).masked_fill(~mask, 0).square()
    pv_mse = squared_error.sum(dim=-1) / counts.clamp(min=1)
    cycle_mse = pv_mse.sum(dim=1) / observed.sum(dim=1).clamp(min=1)
    return cycle_mse, pv_mse, observed


## 4. Train on complete cycle examples

Adam, validation-based learning-rate reduction, and early stopping select the
best validation loss. Batch size is measured in cycles; four cycles already
include up to 144 full waveforms. Raw arrays stay in CPU memory and only the
current batch is transferred to the model device. Training-target loss and
evaluation both use the same masked native-sample MSE.

A matching `cycle_ae.pt` is reused when present. Choose a new `OUTPUT` directory
to train from scratch. Old per-waveform and 256-point weights are incompatible.


In [5]:
def load_cycle_model(directory, device="cpu", expected_signature=None):
    directory = Path(directory)
    config = json.loads((directory / "configuration.json").read_text())
    if config.get("format") != "joint_native_cycle_v1":
        raise ValueError("Expected a joint native-cycle checkpoint")
    if expected_signature is not None and config["signature"] != expected_signature:
        raise ValueError("Checkpoint and dataset/preprocessing differ")
    saved = torch.load(directory / "cycle_ae.pt", map_location=device, weights_only=False)
    if saved["signature"] != config["signature"]:
        raise ValueError("Checkpoint and configuration differ")
    model = CycleWaveformAE(len(config["pvs"]), config["latent"]).to(device)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), config


def train_cycle_model(data):
    data.save_preprocessing()
    path = data.output / "cycle_ae.pt"
    if path.exists():
        model, _ = load_cycle_model(data.output, data.device, data.config["signature"])
        print("Reusing saved cycle model:", path)
        return model
    torch.manual_seed(data.args.seed)
    data.rng = np.random.default_rng(data.args.seed)
    model = CycleWaveformAE(len(data.pvs), data.args.latent).to(data.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=data.args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=.5, patience=3, min_lr=1e-5)
    best, best_state, stale, history = float("inf"), None, 0, []
    started = time.monotonic()
    for epoch in range(1, data.args.epochs + 1):
        record = dict(epoch=epoch)
        for split in ["train", "validation"]:
            model.train(split == "train")
            total, count = 0., 0
            with torch.set_grad_enabled(split == "train"):
                for indices in data.batches(data.splits[split], shuffle=split == "train"):
                    target, mask, _ = data.pack(indices)
                    with torch.autocast(data.device.type, dtype=torch.bfloat16,
                                        enabled=data.device.type == "cuda"):
                        prediction = model(target, mask)
                    scores, _, _ = reconstruction_errors(prediction, target, mask)
                    loss = scores.mean()
                    if not torch.isfinite(loss):
                        raise ValueError(f"Nonfinite loss in epoch {epoch}")
                    if split == "train":
                        optimizer.zero_grad(set_to_none=True)
                        loss.backward()
                        optimizer.step()
                    total += scores.detach().sum().item()
                    count += len(indices)
            if count == 0:
                raise ValueError(f"Empty {split} partition")
            record[f"{split}_mse"] = total / count
        record["learning_rate"] = optimizer.param_groups[0]["lr"]
        record["elapsed_seconds"] = time.monotonic() - started
        history.append(record)
        scheduler.step(record["validation_mse"])
        significant = record["validation_mse"] < best * (1 - 1e-4)
        stale = 0 if significant else stale + 1
        if record["validation_mse"] < best:
            best = record["validation_mse"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save(dict(state_dict=best_state, epoch=epoch, signature=data.config["signature"]), path)
        pd.DataFrame(history).to_csv(data.output / "cycle_ae_history.csv", index=False)
        print(f"Epoch {epoch}: train={record['train_mse']:.6g}, validation={record['validation_mse']:.6g}", flush=True)
        if stale >= data.args.patience:
            break
    model.load_state_dict(best_state)
    history = pd.DataFrame(history)
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    ax.plot(history.epoch, history.train_mse, label="Training")
    ax.plot(history.epoch, history.validation_mse, label="Validation")
    ax.set(xlabel="Epoch", ylabel="Masked native cycle MSE", yscale="log")
    ax.legend()
    fig.savefig(data.output / "learning_curves.png", dpi=150)
    plt.close(fig)
    return model.eval()


## 5. Score cycles and rank PV reconstruction errors

The threshold is fitted only on calibration cycles: median cycle score plus
five population standard deviations (`ddof=0`). A cycle is an anomaly candidate
if its mean observed-PV error exceeds this threshold. PV errors are ranked to
help inspect a flagged cycle; they do not by themselves establish faulty PVs.

Coverage changes the evidence available to the model. Predictions include the
number of observed/missing PVs and are also summarized by coverage. The candidate
rate on these normal candidates is not a validated false-positive rate.


In [6]:
@torch.no_grad()
def score_cycle_records(model, records, pvs, means, stds, device="cpu", batch_size=4):
    model.eval()
    cycle_scores = np.empty(len(records), dtype=np.float64)
    pv_scores = np.full((len(records), len(pvs)), np.nan, dtype=np.float64)
    lengths = np.array([record["length"] for record in records])
    for length in sorted(set(lengths)):
        group = np.flatnonzero(lengths == length)
        for start in range(0, len(group), batch_size):
            indices = group[start:start + batch_size]
            target, mask, _ = pack_cycles([records[i] for i in indices], pvs, means, stds, device)
            prediction = model(target, mask)
            cycle_mse, pv_mse, observed = reconstruction_errors(prediction, target, mask)
            cycle_scores[indices] = cycle_mse.cpu().numpy()
            pv_scores[indices] = pv_mse.masked_fill(~observed, float("nan")).cpu().numpy()
    if not np.isfinite(cycle_scores).all():
        raise ValueError("Nonfinite cycle scores")
    return cycle_scores, pv_scores


def prediction_tables(records, pvs, scores, pv_scores, threshold):
    cycles, pv_rows = [], []
    for i, record in enumerate(records):
        ranked = np.argsort(-np.nan_to_num(pv_scores[i], nan=-np.inf))
        ranked = [j for j in ranked if np.isfinite(pv_scores[i, j])]
        row = dict(cycle_id=record["cycle_id"], cycle_score=float(scores[i]),
                   threshold=float(threshold), anomaly_candidate=bool(scores[i] > threshold),
                   observed_pvs=len(ranked), missing_pvs=len(pvs) - len(ranked),
                   highest_error_pv=pvs[ranked[0]],
                   top5_pvs="|".join(pvs[j] for j in ranked[:5]))
        for column in ["cycle_ts", "beam_mode"]:
            if column in record:
                row[column] = record[column]
        cycles.append(row)
        for rank, j in enumerate(ranked, start=1):
            pv_rows.append(dict(cycle_id=record["cycle_id"], pv=pvs[j], rank=rank,
                                normalized_mse=float(pv_scores[i, j]),
                                observed_samples=len(record["waveforms"][pvs[j]])))
    return pd.DataFrame(cycles), pd.DataFrame(pv_rows)


def evaluate_cycle_model(data, model):
    scores, pv_scores = score_cycle_records(model, data.records, data.pvs, data.means,
                                            data.stds, data.device, data.args.batch_size)
    calibration = scores[data.splits["calibration"]]
    if not len(calibration):
        raise ValueError("No independent calibration cycles")
    median, sigma = float(np.median(calibration)), float(np.std(calibration, ddof=0))
    threshold = median + 5 * sigma
    cycles, pv_errors = prediction_tables(data.records, data.pvs, scores, pv_scores, threshold)
    cycles["split"] = data.meta.split.to_numpy()
    pv_errors = pv_errors.merge(data.meta[["cycle_id", "split"]], on="cycle_id", validate="many_to_one")
    test = cycles.loc[cycles.split.eq("test")]
    metrics = dict(signature=data.config["signature"], calibration_median=median,
                   calibration_std=sigma, threshold=threshold, test_cycles=len(test),
                   test_cycle_mse=float(test.cycle_score.mean()),
                   test_candidate_cycles=int(test.anomaly_candidate.sum()),
                   test_candidate_rate=float(test.anomaly_candidate.mean()))
    data.save_preprocessing()
    cycles.to_csv(data.output / "cycle_predictions.csv", index=False)
    pv_errors.to_csv(data.output / "pv_reconstruction_errors.csv", index=False)
    np.savez(data.output / "cycle_scores.npz", cycle_score=scores, pv_mse=pv_scores)
    write_json(data.output / "cycle_ae_metrics.json", metrics)
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    for split in ["calibration", "test"]:
        ax.hist(np.log10(np.maximum(cycles.loc[cycles.split.eq(split), "cycle_score"], 1e-14)),
                bins=25, alpha=.5, label=split)
    ax.axvline(np.log10(max(threshold, 1e-14)), color="tab:red", ls="--", label="Cycle threshold")
    ax.set(xlabel="log10(masked native cycle MSE)", ylabel="Cycles")
    ax.legend()
    fig.savefig(data.output / "score_distributions.png", dpi=150)
    plt.close(fig)
    return cycles, pv_errors, metrics


## 6. Run training and evaluation

Both training and evaluation process one whole cycle per example. The default
display-only mode does not load the 1.8 GB dataset or require existing results.


In [7]:
data = model = None
if RUN_TRAINING or RUN_EVALUATION:
    torch.set_num_threads(8)
    data = CycleData(args)
    if RUN_TRAINING:
        model = train_cycle_model(data)
    else:
        model, _ = load_cycle_model(OUTPUT, args.device, data.config["signature"])
    cycle_predictions, pv_errors, metrics = evaluate_cycle_model(data, model)
    display(pd.DataFrame([metrics]))
elif (OUTPUT / "cycle_ae_metrics.json").exists():
    metrics = json.loads((OUTPUT / "cycle_ae_metrics.json").read_text())
    cycle_predictions = pd.read_csv(OUTPUT / "cycle_predictions.csv")
    pv_errors = pd.read_csv(OUTPUT / "pv_reconstruction_errors.csv")
    display(pd.DataFrame([metrics]))
else:
    cycle_predictions, pv_errors = pd.DataFrame(), pd.DataFrame()
    print("No joint cycle model has been trained yet. Set RUN_TRAINING = True and run all cells.")


No joint cycle model has been trained yet. Set RUN_TRAINING = True and run all cells.


## 7. Learning curves, cycle decisions, and PV contributions

Saved predictions retain every observed PV error. The table below shows test
cycles, calibration/test coverage summaries, and the highest-ranked PVs for
flagged test cycles. Missing channels never appear as measured PV errors.


In [8]:
for filename in ["learning_curves.png", "score_distributions.png"]:
    if (OUTPUT / filename).exists():
        display(Image(filename=str(OUTPUT / filename)))
if not cycle_predictions.empty:
    display(cycle_predictions.loc[cycle_predictions.split.eq("test")].sort_values("cycle_score", ascending=False))
    display(cycle_predictions.groupby(["split", "observed_pvs"]).agg(
        cycles=("cycle_id", "size"), mean_cycle_mse=("cycle_score", "mean"),
        candidate_cycles=("anomaly_candidate", "sum")))
    flagged = cycle_predictions.loc[cycle_predictions.split.eq("test") & cycle_predictions.anomaly_candidate, "cycle_id"]
    display(pv_errors.loc[pv_errors.cycle_id.isin(flagged) & pv_errors["rank"].le(5)])


## 8. Reconstruct all observed PVs of one cycle

Select a test cycle below. Each plot uses every original sample and crops the
reconstruction to that PV's observed length. Values are restored to recorded
units after applying the training-only PV normalization.


In [9]:
@torch.no_grad()
def reconstruct_cycle(model, record, config, device="cpu"):
    target, mask, lengths = pack_cycles([record], config["pvs"],
                                        config["mean"], config["std"], device)
    model.eval()
    prediction = model(target, mask)[0].cpu().numpy()
    return {pv: prediction[j, :lengths[0, j]] * config["std"][j] + config["mean"][j]
            for j, pv in enumerate(config["pvs"]) if lengths[0, j]}


SELECTED_CYCLE_ID = None  # Set an integer cycle_id; None selects the first test cycle
if data is not None:
    index = int(max(data.splits["test"], key=lambda i: len(data.records[i]["waveforms"]))) if SELECTED_CYCLE_ID is None else next(
        i for i, record in enumerate(data.records) if record["cycle_id"] == SELECTED_CYCLE_ID)
    record = data.records[index]
    reconstructed = reconstruct_cycle(model, record, data.config, data.device)
    fig, axes = plt.subplots(len(reconstructed), 1, figsize=(12, 2.5 * len(reconstructed)),
                             squeeze=False, layout="constrained")
    for ax, (pv, values) in zip(axes[:, 0], reconstructed.items()):
        ax.plot(record["waveforms"][pv], lw=.7, label="Original")
        ax.plot(values, lw=.7, alpha=.8, label="Joint-cycle reconstruction")
        ax.set(title=f"Cycle {record['cycle_id']} | {pv}", xlabel="Native sample index", ylabel="Recorded value")
        ax.legend()
    fig.savefig(OUTPUT / "cycle_reconstruction.png", dpi=120)
    plt.show()
    plt.close(fig)
elif (OUTPUT / "cycle_reconstruction.png").exists():
    display(Image(filename=str(OUTPUT / "cycle_reconstruction.png")))


## 9. Detect anomalies in new cycles

`predict_cycles(frame)` accepts a DataFrame with `cycle_id`, `pv`, and
`data_array` (the native waveform). Supply **all available PV rows for each
cycle together**. The saved PV channel order, normalization, and calibrated
cycle threshold are reused. Optional `cycle_ts` and `beam_mode` are carried
through. Unknown PVs or duplicate captures raise an error.

Returns cycle anomaly decisions and ranked PV reconstruction errors. A high
PV error is a contribution to inspect, not a confirmed PV fault label.


In [10]:
def predict_cycles(frame, directory=OUTPUT, device=None, batch_size=4):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, config = load_cycle_model(directory, device)
    metrics = json.loads((Path(directory) / "cycle_ae_metrics.json").read_text())
    if metrics["signature"] != config["signature"]:
        raise ValueError("Calibration and model configuration differ")
    records, pvs = prepare_cycles(frame, config["pvs"])
    scores, pv_scores = score_cycle_records(model, records, pvs, config["mean"], config["std"],
                                            device, batch_size)
    return prediction_tables(records, pvs, scores, pv_scores, metrics["threshold"])


# Example after training (new_frame must include every available PV of each cycle):
# new_frame = pd.read_pickle(ROOT / "new_mbl_cycles.pkl")
# decisions, ranked_pv_errors = predict_cycles(new_frame)
# display(decisions.loc[decisions.anomaly_candidate])
# display(ranked_pv_errors.loc[ranked_pv_errors["rank"].le(5)])
